In [ ]:
# =============================================================================
# BOOTSTRAP: instalacion automatica de dependencias (self-contained, Linux-ready)
# =============================================================================
import importlib
import subprocess
import sys


def _bootstrap_dependencias(paquetes):
    faltantes = []
    for modulo, paquete_pip in paquetes.items():
        try:
            importlib.import_module(modulo)
        except ImportError:
            faltantes.append(paquete_pip)
    if faltantes:
        print(f"[INFO] Instalando dependencias faltantes: {', '.join(faltantes)}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltantes])


_bootstrap_dependencias({
    "pandas": "pandas",
    "numpy": "numpy",
    "sqlalchemy": "sqlalchemy",
    "pyodbc": "pyodbc",
})

# hasattr(...): bajo Jupyter/ipykernel, sys.stdout/stderr son un OutStream
# propio que NO tiene .reconfigure() (solo lo tiene un TextIOWrapper real,
# el caso de un script corrido por CPython en terminal/cron/systemd) -- sin
# este chequeo, la celda revienta con AttributeError apenas se corre como
# notebook en un kernel.
for _stream in (sys.stdout, sys.stderr):
    if hasattr(_stream, "reconfigure"):
        _stream.reconfigure(encoding="utf-8", errors="replace")

import os
import hashlib
import datetime
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text, bindparam
from urllib.parse import quote_plus


In [ ]:
# =============================================================================
# CONEXIONES Y CREDENCIALES (autocontenido: sin dependencias de otro archivo)
# =============================================================================

def _verificar_driver_odbc(nombre_driver="ODBC Driver 17 for SQL Server"):
    """
    El driver ODBC de SQL Server es un paquete de sistema operativo, no
    instalable por pip. Se valida su presencia antes de conectar para fallar
    con un mensaje claro en vez de un traceback criptico de pyodbc.
    """
    import pyodbc
    disponibles = pyodbc.drivers()
    if nombre_driver not in disponibles:
        raise RuntimeError(
            f"No se encontro el driver ODBC '{nombre_driver}' en este servidor.\n"
            f"Drivers disponibles: {disponibles or '(ninguno)'}\n"
            "Instalar en Debian/Ubuntu:\n"
            "  curl -sSL https://packages.microsoft.com/keys/microsoft.asc | sudo apt-key add -\n"
            "  curl -sSL https://packages.microsoft.com/config/ubuntu/22.04/prod.list | sudo tee /etc/apt/sources.list.d/mssql-release.list\n"
            "  sudo apt-get update && ACCEPT_EULA=Y sudo apt-get install -y msodbcsql17 unixodbc-dev"
        )


_verificar_driver_odbc()

# --- MOTORES DE CONEXION ---
# fast_executemany=True: pyodbc manda cada lote de INSERT en un solo viaje al
# servidor en vez de una ida y vuelta por fila. Es la diferencia entre minutos
# y decenas de minutos al escribir una semana de Scanntech.
conn_str_work = quote_plus(
    "DRIVER=ODBC Driver 17 for SQL Server;"
    "SERVER=10.1.82.27,1433;"
    "DATABASE=SANDBOX_BRONZE;"
    "UID=sa;"
    "PWD=M4rfr1g.2023*;"
    "TrustServerCertificate=yes;"
)
engine_work = create_engine(f"mssql+pyodbc:///?odbc_connect={conn_str_work}", fast_executemany=True)

conn_str_mds = quote_plus(
    "DRIVER=ODBC Driver 17 for SQL Server;"
    "SERVER=10.1.82.27\\mfgbi;"
    "DATABASE=MDS_PRODU;"
    "UID=pbi_report;"
    "PWD=Marfrig2024;"
    "TrustServerCertificate=yes;"
)
engine_mds = create_engine(f"mssql+pyodbc:///?odbc_connect={conn_str_mds}")


In [ ]:
# =============================================================================
# DETECCION DE OUTLIERS - SCANNTECH (COPI)
# -----------------------------------------------------------------------------
# Misma formula que SNP (acordada con analista, 23/04/2026):
#   Limite superior: min(p50 + 8xIQR, p50x2.5)
#   Limite inferior: max(p50 - 8xIQR, p50x0.25)
#
# COPI fue validada OK en Bronze/Silver/Gold por el area de Pricing (mail
# 13/07/2026): esta formula no se toca aca, a diferencia de SNP donde el
# mismo mail reporta falsos positivos (ver silver_snp.ipynb).
#
# Columna de referencia: Precio_Dinamizado (= Precio_Unitario).
# Agrupacion por EAN (= EAN_Final en silver_scanntech) + Fecha.
# Requiere minimo 4 registros por grupo.
#
# Salida: silver_scanntech (validos) + silver_scanntech_outliers (excluidos).
# =============================================================================

def calcular_limites_outlier_scanntech(serie):
    """
    Formula asimetrica p50 + IQR. Retorna (lim_inf, lim_sup, p50, iqr).
    """
    q1  = serie.quantile(0.25)
    q3  = serie.quantile(0.75)
    p50 = serie.quantile(0.50)
    iqr = q3 - q1
    lim_sup = min(p50 + 8 * iqr, p50 * 2.5)
    lim_inf = max(p50 - 8 * iqr, p50 * 0.25)
    return lim_inf, lim_sup, p50, iqr


def detectar_outliers_scanntech(df):
    """
    Deteccion de outliers sobre Precio_Dinamizado (= Precio_Unitario) agrupando
    por EAN + Fecha. En silver_scanntech, EAN ya es EAN_Final (canonico),
    por lo que no hay riesgo de inconsistencias por reemplazos de EAN.

    Columnas de salida:
        _es_outlier    : True si el precio es outlier o nulo/cero.
        _motivo_outlier: descripcion del motivo.
        _lim_inf       : limite inferior del grupo EAN/Fecha.
        _lim_sup       : limite superior del grupo EAN/Fecha.
        _p50           : mediana del grupo (para diagnostico).
        _iqr           : rango intercuartilico del grupo (para diagnostico).
    """
    precios    = df['Precio_Dinamizado']
    sin_precio = precios.isna() | (precios <= 0)

    limites = {
        k: calcular_limites_outlier_scanntech(g['Precio_Dinamizado'].dropna())
        for k, g in df.groupby(['EAN', 'Fecha'])
        if len(g['Precio_Dinamizado'].dropna()) >= 4
    }

    keys = list(zip(df['EAN'], df['Fecha']))
    df['_lim_inf'] = [limites[k][0] if k in limites else None for k in keys]
    df['_lim_sup'] = [limites[k][1] if k in limites else None for k in keys]
    df['_p50']     = [limites[k][2] if k in limites else None for k in keys]
    df['_iqr']     = [limites[k][3] if k in limites else None for k in keys]

    tiene_limites = df['_lim_inf'].notna()

    df['_es_outlier'] = (
        sin_precio |
        (tiene_limites & (precios < df['_lim_inf'])) |
        (tiene_limites & (precios > df['_lim_sup']))
    )

    df['_motivo_outlier'] = pd.Series(pd.NA, index=df.index, dtype='object')
    df.loc[sin_precio, '_motivo_outlier'] = 'Sin precio'
    df.loc[df['_es_outlier'] & ~sin_precio, '_motivo_outlier'] = 'Outlier por EAN/Fecha'

    return df


In [ ]:
# =============================================================================
# CARGA INCREMENTAL POR SEMANA
# -----------------------------------------------------------------------------
# Antes: cada corrida leia TODO PRC.Scanntech_VTA, recalculaba todo y
# reemplazaba silver_scanntech y silver_scanntech_outliers enteras (DROP +
# CREATE + INSERT de millones de filas en chunks de 1000, todas las noches).
#
# Ahora la unidad de trabajo es la SEMANA (Fecha = lunes normalizado):
#   1. Nada en silver cruza semanas: los limites de outlier se calculan por
#      EAN + Fecha, y el resto de las transformaciones son por fila. Rehacer
#      una semana nunca cambia el resultado de otra.
#   2. 01_bronze_scanntech deja en PRC.Scanntech_semanas_origen que ZIP y en
#      que momento cargo cada semana de bronze. Silver guarda en
#      PRC.silver_scanntech_origen la version (archivo + fecha_carga) que
#      proceso de cada semana: toda semana nueva, recargada o eliminada en
#      bronze queda pendiente. Es comparacion de estado, no marca de agua:
#      no depende de relojes ni del orden de las corridas.
#   3. Varias semanas de bronze pueden caer en el mismo lunes (borde entre
#      la era "lunes" 2021-2024 y la era "sabado" 2025-2026). Por eso el
#      pendiente se expande a TODAS las semanas de bronze que caen en los
#      lunes afectados antes de leer y recalcular.
#   4. DELETE de los lunes afectados + INSERT + actualizacion de control van
#      en UNA transaccion: si algo falla a mitad, silver queda como estaba.
#   5. PRC.silver_scanntech_semanas (Fecha -> fecha_proceso) es la senal que
#      consume gold_scanntech_snp para saber que semanas rehacer.
#
# Cuando se rehace TODO (full rebuild):
#   - HORUS_FULL_REBUILD=1 en el entorno (Execute shell de Jenkins).
#   - Cambio en los maestros MDS (EANs / PDVs): un reemplazo de EAN o una
#     correccion de region afecta a toda la historia, y el incremental solo
#     tocaria las semanas nuevas. Se guarda un fingerprint de los maestros en
#     PRC.horus_control y se compara en cada corrida.
#   - Cambio de VERSION_LOGICA (subirla al cambiar una regla de negocio).
#   - Primera corrida / tablas de control o de silver inexistentes.
#
# HORUS_DRY_RUN=1: muestra el plan (que semanas y por que) sin escribir nada.
# =============================================================================

SCHEMA           = 'PRC'
TABLA_BRONZE     = 'Scanntech_VTA'
TABLA_ORIGEN_BZ  = 'Scanntech_semanas_origen'    # la mantiene 01_bronze_scanntech
TABLA_SILVER     = 'silver_scanntech'
TABLA_OUTLIERS   = 'silver_scanntech_outliers'
TABLA_PDVS_SIN   = 'pdvs_sin_maestro_copi'
TABLA_ORIGEN     = 'silver_scanntech_origen'     # semana bronze -> version procesada por silver
TABLA_SEMANAS    = 'silver_scanntech_semanas'    # Fecha (lunes) -> fecha_proceso (la consume gold)
TABLA_CONTROL    = 'horus_control'               # clave/valor compartida por el pipeline
CLAVE_MAESTROS   = 'silver_scanntech.maestros'
VERSION_LOGICA   = '2026-11-17'                  # subir al cambiar reglas -> fuerza full rebuild
CHUNK_SIZE       = 20_000


def _flag_env(nombre):
    return os.environ.get(nombre, '').strip().lower() in ('1', 'true', 'yes', 'si')


FULL_REBUILD_ENV = _flag_env('HORUS_FULL_REBUILD')
DRY_RUN          = _flag_env('HORUS_DRY_RUN')


def semana_a_fecha(serie):
    """
    Semana_Inicio_Semana (entero YYYYMMDD) -> lunes de la semana relevada.
    Origen Scanntech: cae LUNES en la era 2021-2024 (+2 = miercoles) y SABADO
    en la era 2025-2026 (+2 = lunes). Se mantiene el +2 historico y se snapea
    al lunes: era 1 miercoles->lunes (misma semana), era 2 sin cambio.
    """
    f = pd.to_datetime(serie.astype('int64').astype(str), format='%Y%m%d') + pd.Timedelta(days=2)
    return (f - pd.to_timedelta(f.dt.weekday, unit='D')).dt.normalize()


def fecha_de_semana(semana):
    return semana_a_fecha(pd.Series([int(semana)])).iloc[0].date()


def fingerprint_maestros(*dfs):
    """Hash estable del contenido de los maestros + version de la logica."""
    h = hashlib.sha256(VERSION_LOGICA.encode())
    for df in dfs:
        d = df.astype(str)
        d = d.sort_values(list(d.columns)).reset_index(drop=True)
        h.update(pd.util.hash_pandas_object(d, index=False).values.tobytes())
    return h.hexdigest()


def _in_clause(sql, nombre):
    return text(sql).bindparams(bindparam(nombre, expanding=True))


def tabla_existe(conn, tabla):
    return conn.execute(text("SELECT OBJECT_ID(:n, 'U')"), {'n': f'{SCHEMA}.{tabla}'}).scalar() is not None


def asegurar_tablas_control(conn):
    conn.execute(text(f"""
        IF OBJECT_ID('[{SCHEMA}].[{TABLA_ORIGEN}]', 'U') IS NULL
        CREATE TABLE [{SCHEMA}].[{TABLA_ORIGEN}] (
            Semana_Inicio_Semana BIGINT PRIMARY KEY,
            nombre_archivo       NVARCHAR(500) NULL,
            fecha_carga_bronze   DATETIME NULL,
            Fecha                DATE NOT NULL,
            fecha_proceso        DATETIME DEFAULT GETDATE()
        )
    """))
    conn.execute(text(f"""
        IF OBJECT_ID('[{SCHEMA}].[{TABLA_SEMANAS}]', 'U') IS NULL
        CREATE TABLE [{SCHEMA}].[{TABLA_SEMANAS}] (
            Fecha          DATE PRIMARY KEY,
            filas          INT,
            filas_outliers INT,
            fecha_proceso  DATETIME DEFAULT GETDATE()
        )
    """))
    conn.execute(text(f"""
        IF OBJECT_ID('[{SCHEMA}].[{TABLA_CONTROL}]', 'U') IS NULL
        CREATE TABLE [{SCHEMA}].[{TABLA_CONTROL}] (
            clave NVARCHAR(200) PRIMARY KEY,
            valor NVARCHAR(MAX),
            fecha DATETIME DEFAULT GETDATE()
        )
    """))


def leer_control(conn, clave):
    return conn.execute(text(f"SELECT valor FROM [{SCHEMA}].[{TABLA_CONTROL}] WHERE clave = :k"),
                        {'k': clave}).scalar()


def escribir_control(conn, clave, valor):
    conn.execute(text(f"DELETE FROM [{SCHEMA}].[{TABLA_CONTROL}] WHERE clave = :k"), {'k': clave})
    conn.execute(text(f"INSERT INTO [{SCHEMA}].[{TABLA_CONTROL}] (clave, valor) VALUES (:k, :v)"),
                 {'k': clave, 'v': valor})


def leer_origen_bronze(conn):
    """{semana: (nombre_archivo, fecha_carga)} segun 01_bronze_scanntech."""
    if not tabla_existe(conn, TABLA_ORIGEN_BZ):
        raise RuntimeError(
            f"No existe [{SCHEMA}].[{TABLA_ORIGEN_BZ}]: correr primero la version actual de "
            "01_bronze_scanntech, que es la que registra el origen de cada semana de bronze."
        )
    rows = conn.execute(text(
        f"SELECT Semana_Inicio_Semana, nombre_archivo, fecha_carga FROM [{SCHEMA}].[{TABLA_ORIGEN_BZ}]"
    )).fetchall()
    return {int(r[0]): (r[1], r[2]) for r in rows}


def leer_origen_silver(conn):
    """{semana: (nombre_archivo, fecha_carga_bronze, Fecha)} procesado por silver."""
    rows = conn.execute(text(
        f"SELECT Semana_Inicio_Semana, nombre_archivo, fecha_carga_bronze, Fecha FROM [{SCHEMA}].[{TABLA_ORIGEN}]"
    )).fetchall()
    return {int(r[0]): (r[1], r[2], r[3]) for r in rows}


def determinar_pendientes(origen_bz, origen_sv):
    """
    Devuelve (cambiadas, eliminadas, motivos):
      cambiadas : semanas de bronze nuevas o cuya version (archivo, fecha_carga)
                  difiere de la que silver proceso.
      eliminadas: semanas que silver proceso y ya no estan en bronze.
    """
    cambiadas, motivos = [], {}
    for s in sorted(origen_bz):
        nombre, fc = origen_bz[s]
        prev = origen_sv.get(s)
        if prev is None:
            motivo = 'nueva en bronze'
        elif (prev[0], prev[1]) != (nombre, fc):
            motivo = f'recargada en bronze ({prev[0]} @ {prev[1]} -> {nombre} @ {fc})'
        else:
            continue
        cambiadas.append(s)
        motivos[s] = motivo
    eliminadas = sorted(s for s in origen_sv if s not in origen_bz)
    return cambiadas, eliminadas, motivos


def expandir_a_fechas(cambiadas, eliminadas, origen_bz, origen_sv):
    """
    Lunes afectados por las semanas pendientes, y TODAS las semanas de bronze
    que caen en esos lunes (para recalcular cada lunes con su contenido completo).
    """
    fechas = set()
    for s in cambiadas:
        fechas.add(fecha_de_semana(s))
        if s in origen_sv:
            fechas.add(origen_sv[s][2])
    for s in eliminadas:
        fechas.add(origen_sv[s][2])
    semanas_leer = sorted(s for s in origen_bz if fecha_de_semana(s) in fechas)
    return sorted(fechas), semanas_leer


def leer_bronze(semanas=None):
    """Lee bronze completo (semanas=None) o solo las semanas indicadas."""
    if semanas is None:
        return pd.read_sql(f"SELECT * FROM [{SCHEMA}].[{TABLA_BRONZE}]", engine_work)
    if not semanas:
        return pd.read_sql(f"SELECT TOP 0 * FROM [{SCHEMA}].[{TABLA_BRONZE}]", engine_work)
    partes = []
    with engine_work.connect() as conn:
        for i in range(0, len(semanas), 500):
            stmt = _in_clause(
                f"SELECT * FROM [{SCHEMA}].[{TABLA_BRONZE}] WHERE Semana_Inicio_Semana IN :semanas", 'semanas')
            partes.append(pd.read_sql(stmt, conn, params={'semanas': list(semanas[i:i + 500])}))
    return pd.concat(partes, ignore_index=True)


def _a_datetime(fechas):
    return [datetime.datetime.combine(f, datetime.time()) for f in fechas]


def borrar_fechas(conn, tabla, fechas, columna='Fecha', extra_sql=''):
    """DELETE por lista de fechas, en lotes (limite de parametros de SQL Server)."""
    if not fechas:
        return
    for i in range(0, len(fechas), 500):
        stmt = _in_clause(f"DELETE FROM [{SCHEMA}].[{tabla}] WHERE {columna} IN :fechas{extra_sql}", 'fechas')
        conn.execute(stmt, {'fechas': fechas[i:i + 500]})


def crear_tabla_desde_df(conn, df, tabla):
    """
    Crea (o recrea) la tabla con el esquema inferido del DataFrame COMPLETO
    (antes se infería del primer chunk de 1000 filas) y un indice clustered por
    Fecha, que es lo que hace rapido el DELETE por semana del incremental.
    """
    df.head(0).to_sql(tabla, con=conn, schema=SCHEMA, if_exists='replace', index=False)
    conn.execute(text(f"CREATE CLUSTERED INDEX [IX_{tabla}_Fecha] ON [{SCHEMA}].[{tabla}] (Fecha)"))


def insertar(conn, df, tabla):
    total = len(df)
    print(f"[INFO] Insertando {total:,} filas en {SCHEMA}.{tabla}...")
    if total:
        df.to_sql(tabla, con=conn, schema=SCHEMA, if_exists='append', index=False, chunksize=CHUNK_SIZE)
    print(f"[OK] {SCHEMA}.{tabla}: {total:,} filas.")


In [ ]:
# =============================================================================
# TRANSFORMACION (por fila y por EAN+Fecha: no cruza semanas)
# =============================================================================

def leer_maestros():
    print("[INFO] Extrayendo maestros de MDS_PRODU...")
    df_eans = pd.read_sql("""
        SELECT
            EAN, Reemplazo, [KG / U],
            Familia, Empresa, Marca, Segmento,
            [Cód SKU Marfrig]        AS Cod_SKU,
            [Descripción SKU Nuevo]  AS Descripcion_SKU,
            [Marca Propia_Name]      AS Marca_Propia
        FROM [mdm].[ICO_EANs_mercado]
    """, engine_mds)

    df_pdvs = pd.read_sql("""
        SELECT
            [Código único PDV],
            [Clúster Scanntech],
            [Nombre PDV]    AS Nombre_PDV,
            QF_Provincia    AS Provincia,
            Región          AS Region,
            Localidad,
            [Clúster Para]  AS Cluster_Para,
            [Subcanal Para] AS Subcanal
        FROM [mdm].[ICO_PDVs_Scanntech]
    """, engine_mds)
    return df_eans, df_pdvs


def transformar(df_vta, df_eans, df_pdvs):
    """
    Bronze -> silver para el conjunto de filas recibido. Devuelve
    (df_validos, df_outliers, pdvs_sin_maestro).
    """
    df_eans = df_eans.copy()
    df_pdvs = df_pdvs.copy()

    # A. Normalizacion de Fecha -> LUNES de la semana relevada (A1).
    df_vta['Fecha'] = semana_a_fecha(df_vta['Semana_Inicio_Semana'])

    # B. Facturacion y Precio Unitario
    # Precio_por_Unidade contiene la FACTURACION (monto total de la venta del periodo).
    # El precio por unidad se calcula dividiendo la facturacion por la cantidad vendida.
    df_vta['Precio_por_Unidade'] = pd.to_numeric(
        df_vta['Precio_por_Unidade'].astype(str).str.replace(',', '.'), errors='coerce'
    )
    df_vta['Facturacion'] = df_vta['Precio_por_Unidade']

    df_vta['Cantidad_de_Venta'] = pd.to_numeric(df_vta['Cantidad_de_Venta'], errors='coerce')
    cantidad = df_vta['Cantidad_de_Venta'].replace(0.0, float('nan'))
    df_vta['Precio_Unitario'] = df_vta['Facturacion'] / cantidad

    # C. Join de EANs (normalizacion + dimensiones de producto)
    # Deduplicar df_eans por EAN para evitar explosion de filas en el merge
    df_vta['Codigo_Barras_SKU'] = df_vta['Codigo_Barras_SKU'].astype(str)
    df_eans['EAN'] = df_eans['EAN'].astype(str)
    df_eans = df_eans.drop_duplicates(subset='EAN', keep='first')

    df_final = pd.merge(df_vta, df_eans, left_on='Codigo_Barras_SKU', right_on='EAN', how='left')

    # Reemplazo: strip + vacio/NaN -> usar EAN original
    # fillna('') ANTES de astype(str): la regla no puede depender de que
    # astype(str) convierta un nulo en el texto 'nan'. Con el dtype de texto
    # nuevo de pandas (por defecto desde la version 3, u opcion
    # future.infer_string) el nulo sigue nulo, el filtro != '' no lo detecta
    # y todo EAN sin reemplazo quedaba con EAN_Final nulo (detectado el
    # 17/11/2026: ~94% de las filas sin EAN en silver y gold).
    df_final['Reemplazo'] = (
        df_final['Reemplazo'].fillna('').astype(str).str.strip()
        .replace({'nan': '', 'None': '', '<NA>': ''})
    )
    df_final['EAN_Final'] = df_final['Reemplazo'].where(
        df_final['Reemplazo'] != '', df_final['Codigo_Barras_SKU']
    )

    # Descripcion_SKU desde EAN_Final (canonico), descartando la del EAN original
    df_desc = (
        df_eans[['EAN', 'Descripcion_SKU']]
        .drop_duplicates(subset='EAN', keep='first')
        .rename(columns={'EAN': 'EAN_Final', 'Descripcion_SKU': '_desc_final'})
    )
    df_final = pd.merge(df_final, df_desc, on='EAN_Final', how='left')
    df_final['Descripcion_SKU'] = df_final['_desc_final'].fillna(df_final['Descripcion_SKU'])
    df_final = df_final.drop(columns=['_desc_final'])

    # D. Calculo de KG y Precio por KG
    # Precio_KG = Precio unitario / (KG por unidad)
    kgu_num = pd.to_numeric(df_final['KG / U'], errors='coerce')
    df_final['KG']        = df_final['Cantidad_de_Venta'] * kgu_num.fillna(0)
    df_final['Precio_KG'] = df_final['Precio_Unitario'] / kgu_num.replace(0.0, float('nan'))

    # E. Join de PDVs (dimensiones de punto de venta)
    df_final['Codigo_Unico_PDV'] = df_final['Codigo_Unico_PDV'].astype(str)
    df_pdvs['Código único PDV'] = df_pdvs['Código único PDV'].astype(str)
    df_final = pd.merge(df_final, df_pdvs, left_on='Codigo_Unico_PDV', right_on='Código único PDV', how='left')
    df_final['Clúster Scanntech'] = df_final['Clúster Scanntech'].fillna('NO')

    # E2. Correccion de Provincia mal cargada en el maestro MDS (trazable).
    # PDV 368411914 (Localidad SALLIQUELO, Region PBA) trae QF_Provincia='CABA'
    # por error; Salliquelo pertenece a la provincia de Buenos Aires. Se reasigna
    # a la Provincia modal de su misma Region para no hardcodear la etiqueta.
    PDVS_PROVINCIA_INCORRECTA = ['368411914']
    _mask_fix = df_final['Codigo_Unico_PDV'].isin(PDVS_PROVINCIA_INCORRECTA)
    if _mask_fix.any():
        _prov_ok = (
            df_final.loc[(df_final['Region'] == 'PBA') & (~_mask_fix), 'Provincia']
            .mode()
        )
        _prov_ok = _prov_ok.iloc[0] if len(_prov_ok) else 'Buenos Aires'
        df_final.loc[_mask_fix, 'Provincia'] = _prov_ok
        print(f"[INFO] Correccion Provincia: {int(_mask_fix.sum()):,} filas de "
              f"{PDVS_PROVINCIA_INCORRECTA} reasignadas a '{_prov_ok}'.")

    # A3. PDVs ausentes del maestro (no se filtran en silencio).
    # Se registran para correccion en MDS y se etiquetan 'Sin asignar' para
    # quedar trazables como bucket explicito aguas abajo (gold/platinum).
    pdvs_sin_maestro = (
        df_final.loc[df_final['Region'].isna(), ['Codigo_Unico_PDV']]
        .drop_duplicates()
        .rename(columns={'Codigo_Unico_PDV': 'Codigo_PDV'})
    )
    n_filas_sin_region = int(df_final['Region'].isna().sum())
    if len(pdvs_sin_maestro) > 0:
        print(f"[WARN] A3: {len(pdvs_sin_maestro):,} PDVs sin Region en maestro "
              f"({n_filas_sin_region:,} filas) -> etiquetados 'Sin asignar'. "
              f"Lista cargada en {SCHEMA}.{TABLA_PDVS_SIN} para MDS.")
    df_final['Region'] = df_final['Region'].fillna('Sin asignar')

    # 4. FORMATEO DE TABLA FINAL
    columnas_salida = {
        'Fecha':              'Fecha',
        'Codigo_Unico_PDV':   'Codigo_PDV',
        'Nombre_PDV':         'Nombre_PDV',
        'Provincia':          'Provincia',
        'Region':             'Region',
        'Localidad':          'Localidad',
        'Clúster Scanntech':  'Cluster_Scanntech',
        'Cluster_Para':       'Cluster_Para',
        'Subcanal':           'Subcanal',
        'EAN_Final':          'EAN',
        'Cod_SKU':            'Cod_SKU',
        'Descripcion_SKU':    'Descripcion_SKU',
        'Familia':            'Familia',
        'Empresa':            'Empresa',
        'Marca':              'Marca',
        'Segmento':           'Segmento',
        'Marca_Propia':       'Marca_Propia',
        'Cantidad_de_Venta':  'Cantidad',
        'Facturacion':        'Facturacion',
        'KG':                 'KG',
        'Precio_Unitario':    'Precio_Dinamizado',
        'Precio_KG':          'Precio_KG',
    }

    columnas_faltantes = [k for k in columnas_salida if k not in df_final.columns]
    if columnas_faltantes:
        print(f"[WARN] Columnas esperadas no encontradas en df_final: {columnas_faltantes}")

    columnas_presentes = {k: v for k, v in columnas_salida.items() if k in df_final.columns}
    df_silver = df_final[list(columnas_presentes.keys())].rename(columns=columnas_presentes)

    # Precio_BASE = Precio_Dinamizado (utilizado para append con SNP en capa Gold)
    df_silver['Precio_BASE'] = df_silver['Precio_Dinamizado']

    # Prefijo de fuente en Codigo_PDV para identificacion en capa Gold
    df_silver['Codigo_PDV'] = 'Scanntech_' + df_silver['Codigo_PDV'].astype(str)

    # Sanitizacion de inf/-inf: SQL Server no acepta estos valores en columnas float
    df_silver = df_silver.replace([float('inf'), float('-inf')], None)

    # 5. DETECCION DE OUTLIERS
    # Mismo criterio que SNP: formula asimetrica p50 + IQR agrupada por EAN + Fecha.
    # En este punto EAN = EAN_Final (ya renombrado), sin riesgo de inconsistencias.
    print("[INFO] Detectando outliers (p50+/-8xIQR / p50x[0.25-2.5], criterio EAN/Fecha)...")
    df_silver = detectar_outliers_scanntech(df_silver)

    n_outliers   = int(df_silver['_es_outlier'].sum())
    n_validos    = int((~df_silver['_es_outlier']).sum())
    pct_outliers = n_outliers / len(df_silver) * 100 if len(df_silver) > 0 else 0
    print(f"   Registros validos:   {n_validos:,}")
    print(f"   Outliers detectados: {n_outliers:,} ({pct_outliers:.2f}%)")

    if n_outliers > 0:
        breakdown = (
            df_silver[df_silver['_es_outlier']]
            .groupby('_motivo_outlier')
            .size()
            .reset_index(name='cantidad')
        )
        print("   Detalle por motivo:")
        print(breakdown.to_string(index=False))

    # 6. SEPARAR VALIDOS Y OUTLIERS
    COLS_OUTLIER = ['_es_outlier', '_motivo_outlier', '_lim_inf', '_lim_sup', '_p50', '_iqr']
    COLS_SILVER  = [c for c in df_silver.columns if c not in COLS_OUTLIER]

    df_validos  = df_silver[~df_silver['_es_outlier']][COLS_SILVER].copy()
    df_outliers = df_silver[ df_silver['_es_outlier']][COLS_SILVER + COLS_OUTLIER].copy()

    return df_validos, df_outliers, pdvs_sin_maestro


In [ ]:
# =============================================================================
# PIPELINE PRINCIPAL
# =============================================================================

def run_silver_pipeline_script():
    print(f"--- INICIO PROCESO SILVER SCANNTECH: {datetime.datetime.now()} ---")
    print(f"[INFO] pandas {pd.__version__} | dtype de texto por defecto: "
          f"{pd.Series(['a']).dtype}")
    try:
        with engine_work.begin() as conn:
            asegurar_tablas_control(conn)

        # 1. MAESTROS + DECISION FULL / INCREMENTAL
        df_eans, df_pdvs = leer_maestros()
        fp_actual = fingerprint_maestros(df_eans, df_pdvs)

        with engine_work.connect() as conn:
            fp_previo     = leer_control(conn, CLAVE_MAESTROS)
            origen_bz     = leer_origen_bronze(conn)
            origen_sv     = leer_origen_silver(conn)
            silver_existe = tabla_existe(conn, TABLA_SILVER) and tabla_existe(conn, TABLA_OUTLIERS)

        motivo_full = None
        if FULL_REBUILD_ENV:
            motivo_full = 'HORUS_FULL_REBUILD=1'
        elif not silver_existe:
            motivo_full = 'tablas silver inexistentes'
        elif not origen_sv:
            motivo_full = 'sin historial de silver (primera corrida incremental)'
        elif fp_previo != fp_actual:
            motivo_full = 'cambio en maestros MDS o en VERSION_LOGICA'
        full = motivo_full is not None

        print(f"\n[INFO] Bronze: {len(origen_bz)} semanas con origen registrado | "
              f"silver: {len(origen_sv)} semanas procesadas.")

        if full:
            print(f"[INFO] Modo: FULL REBUILD ({motivo_full}).")
            fechas, semanas_leer = None, None
            cambiadas = eliminadas = []
        else:
            cambiadas, eliminadas, motivos = determinar_pendientes(origen_bz, origen_sv)
            fechas, semanas_leer = expandir_a_fechas(cambiadas, eliminadas, origen_bz, origen_sv)
            print(f"[INFO] Modo: INCREMENTAL. {len(cambiadas)} semanas cambiadas, "
                  f"{len(eliminadas)} eliminadas en bronze -> {len(fechas)} lunes a rehacer "
                  f"({len(semanas_leer)} semanas de bronze a leer).")
            for s in cambiadas:
                print(f"   - {s} -> lunes {fecha_de_semana(s)}: {motivos[s]}")
            for s in eliminadas:
                print(f"   - {s} -> lunes {origen_sv[s][2]}: ya no esta en bronze, se elimina de silver")
            if not fechas:
                print("\n[OK] Nada pendiente. Silver ya esta al dia.")
                print(f"\n[OK] PROCESO FINALIZADO EXITOSAMENTE: {datetime.datetime.now()}")
                return

        if DRY_RUN:
            print("\n[DRY-RUN] HORUS_DRY_RUN=1: no se escribe nada.")
            return

        # 2. LEER BRONZE (todo o solo las semanas afectadas)
        print("\n[INFO] Extrayendo datos de Bronze...")
        df_vta = leer_bronze(None if full else semanas_leer)
        print(f"   {len(df_vta):,} filas leidas.")

        semanas_vta = sorted(int(s) for s in df_vta['Semana_Inicio_Semana'].dropna().unique())
        if full:
            sin_origen = [s for s in semanas_vta if s not in origen_bz]
            if sin_origen:
                print(f"[WARN] {len(sin_origen)} semanas en bronze sin origen registrado "
                      f"(cargadas por una version anterior de bronze): {sin_origen[:10]}")
            semanas_registrar = sorted(set(semanas_vta) | set(origen_bz))
        else:
            semanas_registrar = semanas_leer

        # 3. TRANSFORMAR
        print("\n[INFO] Aplicando transformaciones...")
        df_validos, df_outliers, pdvs_sin_maestro = transformar(df_vta, df_eans, df_pdvs)
        del df_vta

        if not full:
            # Solo los lunes afectados: las semanas leidas caen todas ahi por
            # construccion, esto es una red de seguridad.
            fechas_dt = set(fechas)
            df_validos  = df_validos[df_validos['Fecha'].dt.date.isin(fechas_dt)]
            df_outliers = df_outliers[df_outliers['Fecha'].dt.date.isin(fechas_dt)]

        # 4. ESCRIBIR (una sola transaccion: silver + outliers + control)
        print("\n[INFO] Escribiendo en base de datos...")
        with engine_work.begin() as conn:
            if full:
                crear_tabla_desde_df(conn, df_validos,  TABLA_SILVER)
                crear_tabla_desde_df(conn, df_outliers, TABLA_OUTLIERS)
                conn.execute(text(f"DELETE FROM [{SCHEMA}].[{TABLA_ORIGEN}]"))
                conn.execute(text(f"DELETE FROM [{SCHEMA}].[{TABLA_SEMANAS}]"))
                pdvs_previos = pd.DataFrame(columns=['Codigo_PDV'])
            else:
                fechas_dt = _a_datetime(fechas)
                borrar_fechas(conn, TABLA_SILVER,   fechas_dt)
                borrar_fechas(conn, TABLA_OUTLIERS, fechas_dt)
                borrar_fechas(conn, TABLA_ORIGEN,  list(set(semanas_registrar) | set(eliminadas)),
                              columna='Semana_Inicio_Semana')
                borrar_fechas(conn, TABLA_SEMANAS, list(fechas))
                pdvs_previos = (pd.read_sql(f"SELECT Codigo_PDV FROM [{SCHEMA}].[{TABLA_PDVS_SIN}]", conn)
                                if tabla_existe(conn, TABLA_PDVS_SIN) else pd.DataFrame(columns=['Codigo_PDV']))

            insertar(conn, df_validos,  TABLA_SILVER)
            insertar(conn, df_outliers, TABLA_OUTLIERS)

            # PDVs sin maestro: se acumulan (merge), no se pisan con los de esta corrida.
            pdvs_todos = (pd.concat([pdvs_previos, pdvs_sin_maestro], ignore_index=True)
                          .astype(str).drop_duplicates())
            pdvs_todos.to_sql(TABLA_PDVS_SIN, con=conn, schema=SCHEMA, if_exists='replace', index=False)

            # Control por semana de bronze (version procesada)
            if semanas_registrar:
                conn.execute(text(f"""
                    INSERT INTO [{SCHEMA}].[{TABLA_ORIGEN}]
                        (Semana_Inicio_Semana, nombre_archivo, fecha_carga_bronze, Fecha)
                    VALUES (:s, :n, :fc, :f)
                """), [{'s': s,
                        'n': origen_bz.get(s, (None, None))[0],
                        'fc': origen_bz.get(s, (None, None))[1],
                        'f': fecha_de_semana(s)} for s in semanas_registrar])

            # Control por lunes (senal para gold)
            conteo_v = df_validos.groupby(df_validos['Fecha'].dt.date).size()
            conteo_o = df_outliers.groupby(df_outliers['Fecha'].dt.date).size()
            fechas_res = sorted(set(conteo_v.index) | set(conteo_o.index))
            if fechas_res:
                conn.execute(text(f"""
                    INSERT INTO [{SCHEMA}].[{TABLA_SEMANAS}] (Fecha, filas, filas_outliers)
                    VALUES (:f, :v, :o)
                """), [{'f': f, 'v': int(conteo_v.get(f, 0)), 'o': int(conteo_o.get(f, 0))} for f in fechas_res])

            escribir_control(conn, CLAVE_MAESTROS, fp_actual)

        print(f"[OK] {len(fechas_res)} lunes escritos en silver"
              + ("" if full else f" (de {len(fechas)} afectados; los sin filas quedaron eliminados)") + ".")
        print(f"\n[OK] PROCESO FINALIZADO EXITOSAMENTE: {datetime.datetime.now()}")

    except Exception as e:
        print(f"[ERROR] ERROR EN EL SCRIPT: {e}")
        raise


if __name__ == "__main__":
    run_silver_pipeline_script()
